# Agriculture Data Cleaning: USGS County Nutrient Inputs from Manure (Falcone 2020)

Cleans the USGS Falcone (2020) county-level **manure** workbook into three tidy long tables. The workbook stores one wide sheet per census year (1950, 1954, ..., 2017); each sheet holds, per county: animal inventory counts, modelled N/P from manure by animal category, and a column of per-year weight coefficients. We split those three concerns into three outputs.

The raw workbook is national; this is an Iowa water-quality study, so the two county-level outputs are filtered to Iowa's 99 counties (state FIPS 19). The weight-coefficient table is a non-spatial national reference and is kept whole.

**Input:**  `data/tabular/01_raw/agriculture/N-P_from_manure_1950-2017-july23-2020.xlsx`
**Outputs:**
- `np-manure-clean.csv` - N/P kg by `(county_fips, year, animal_category, nutrient)`; categories `Cattle, Hogs, Poultry, Other, Total`.
- `manure-animal-inventory-clean.csv` - head counts by `(county_fips, year, animal_type)`.
- `manure-weight-coefficients-clean.csv` - the `coef` reference tab (USDA live slaughter-weight change vs. the 1992 base year), by `(year, animal)`.

Counts in 1950-2012 come from LaMotte (2015)'s Census of Agriculture compilation; 2017 from USDA QuickStats. N/P are Gronberg & Arnold (2017) formulas. Some counts were USDA-suppressed and back-filled by Falcone (the `adj` suffix); we record that in an `adjusted` flag.

**Pipeline**
1. **Load** every year sheet + `coef`. 2. **Guard** the schema. 3. **Melt** nutrient blocks and inventory blocks separately, parsing `year` / category / animal from column names. 4. **Tidy** the `coef` reference. 5. **Key**, check, save.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "agriculture"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "agriculture"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Project scope: this is an Iowa water-quality study, so we narrow the national
# workbook to Iowa (state FIPS 19 / postal "IA", 99 counties) for the output.
# Schema guards still run against the full national pull before we filter.
STATE = "IA"
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)


def add_fips(df: pd.DataFrame) -> pd.DataFrame:
    """Zero-pad STCOFIPS into 5-digit county_fips and derive 2-digit state_fips."""
    df = df.copy()
    df["county_fips"] = df["STCOFIPS"].astype(int).astype(str).str.zfill(5)
    df["state_fips"] = df["county_fips"].str[:2]
    return df.rename(columns={"CountyName": "county_name", "State": "state"})


Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/agriculture
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture


## Step 1 - Load every year sheet and the coefficient tab

In [2]:
RAW_FILE = "N-P_from_manure_1950-2017-july23-2020.xlsx"
xl = pd.ExcelFile(RAW_DIR / RAW_FILE)
YEAR_SHEETS = sorted(s for s in xl.sheet_names if s.isdigit())
print(f"{len(YEAR_SHEETS)} year sheets: {YEAR_SHEETS}")

frames = {int(s): xl.parse(s) for s in YEAR_SHEETS}
frames[1950].head(3)

15 year sheets: ['1950', '1954', '1959', '1964', '1969', '1974', '1978', '1982', '1987', '1992', '1997', '2002', '2007', '2012', '2017']


,STCOFIPS,fips-int,CountyName,State,bcows1950adj,mcows1950adj,cows1950adj,othercows1950,hogs1950adj,chickens1950adj,...,Other_P_kg-1950,Total_N_kg-1950,Total_P_kg-1950,Unnamed: 24,cattle-weight-coef,hogs-weight-coef,chickens-weight-coef,broilers-weight-coef,turkeys-weight-coef,sheep-weight-coef
0,1001,1001,Autauga,AL,6290,2938,17250,8022,15059,51846,...,4350.9460,9.865080e+05,295211.762384,NaN,0.869,0.942,1.039562,0.743538,0.693099,0.78
1,1003,1003,Baldwin,AL,9071,6574,30810,15165,17805,177788,...,10214.1892,1.722134e+06,486365.892105,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1005,1005,Barbour,AL,8079,4976,23756,10701,35068,57674,...,6753.2300,1.495861e+06,455790.993439,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 - Schema guards

Each year sheet is one row per county with unique FIPS. We also pin the column vocabulary: the 10 inventory animals and the 5 nutrient categories x 2 nutrients.

In [3]:
# inventory column = <animal><year>[adj]; "adj" marks a back-filled (suppressed) count
INV_RE = re.compile(r"^(?P<animal>[a-z]+?)(?P<year>\d{4})(?P<adj>adj)?$")
INV_ANIMALS = {"bcows", "mcows", "cows", "othercows", "hogs",
               "chickens", "broilers", "turkeys", "sheep", "horses"}
# nutrient column = <Category>_<N|P>_kg-<year>
NUT_RE = re.compile(r"^(?P<category>[A-Za-z]+)_(?P<nutrient>[NP])_kg-(?P<year>\d{4})$")
NUT_CATS = {"Cattle", "Hogs", "Poultry", "Other", "Total"}

for year, d in frames.items():
    assert len(d) == 3066, f"{year}: expected 3066 rows, got {len(d)}"
    assert not d["STCOFIPS"].duplicated().any(), f"{year}: duplicate county FIPS"
    inv = {INV_RE.match(c)["animal"] for c in d.columns
           if INV_RE.match(str(c)) and INV_RE.match(c)["animal"] in INV_ANIMALS}
    assert inv == INV_ANIMALS, f"{year}: inventory animals {inv} != {INV_ANIMALS}"
    cats = {NUT_RE.match(c)["category"] for c in d.columns if NUT_RE.match(str(c))}
    assert cats == NUT_CATS, f"{year}: nutrient categories {cats} != {NUT_CATS}"
print("Schema guards passed: 3,066 counties/sheet, 10 inventory animals, 5 nutrient categories.")

Schema guards passed: 3,066 counties/sheet, 10 inventory animals, 5 nutrient categories.


## Step 3 - Nutrient table

Melt the `<Category>_<N|P>_kg-<year>` blocks from every sheet into one long table keyed by `(county_fips, year, animal_category, nutrient)`.

In [4]:
def tidy_nutrients(df: pd.DataFrame) -> pd.DataFrame:
    val_cols = [c for c in df.columns if NUT_RE.match(str(c))]
    long = df.melt(
        id_vars=["STCOFIPS", "CountyName", "State"],
        value_vars=val_cols, var_name="col", value_name="value_kg",
    )
    meta = long["col"].str.extract(NUT_RE)
    long["animal_category"] = meta["category"]
    long["nutrient"] = meta["nutrient"]
    long["year"] = meta["year"].astype(int)
    return long.drop(columns="col")

nut = pd.concat([tidy_nutrients(d) for d in frames.values()], ignore_index=True)
nut = add_fips(nut)
nut = nut[nut["state"] == STATE].copy()  # project scope: Iowa only
nut["value_kg"] = pd.to_numeric(nut["value_kg"], errors="coerce")

NUT_COLS = ["state_fips", "county_fips", "county_name", "state",
            "year", "animal_category", "nutrient", "value_kg"]
nut_clean = nut[NUT_COLS].sort_values(
    ["county_fips", "year", "animal_category", "nutrient"]
).reset_index(drop=True)
assert not nut_clean.duplicated(
    ["county_fips", "year", "animal_category", "nutrient"]).any()
print(f"Nutrient table: {len(nut_clean):,} rows")
nut_clean.head()

Nutrient table: 14,850 rows


,state_fips,county_fips,county_name,state,year,animal_category,nutrient,value_kg
0,19,19001,Adair,IA,1950,Cattle,N,2.208110e+06
1,19,19001,Adair,IA,1950,Cattle,P,5.637835e+05
2,19,19001,Adair,IA,1950,Hogs,N,9.186584e+05
3,19,19001,Adair,IA,1950,Hogs,P,4.082926e+05
4,19,19001,Adair,IA,1950,Other,N,2.498593e+05


## Step 4 - Animal-inventory table

Melt the `<animal><year>[adj]` count blocks. We map the terse animal tokens to readable labels (per the `notes` tab) and keep an `adjusted` flag for the back-filled (originally USDA-suppressed) counts.

In [5]:
ANIMAL_LABELS = {
    "bcows": "beef cows", "mcows": "milk cows", "cows": "all cattle and calves",
    "othercows": "other cattle and calves", "hogs": "hogs and pigs",
    "chickens": "layers", "broilers": "broilers", "turkeys": "turkeys",
    "sheep": "sheep and lambs", "horses": "horses and ponies",
}

def tidy_inventory(df: pd.DataFrame) -> pd.DataFrame:
    val_cols = [c for c in df.columns
                if INV_RE.match(str(c)) and INV_RE.match(c)["animal"] in INV_ANIMALS]
    long = df.melt(
        id_vars=["STCOFIPS", "CountyName", "State"],
        value_vars=val_cols, var_name="col", value_name="head_count",
    )
    meta = long["col"].str.extract(INV_RE)
    long["animal"] = meta["animal"].map(ANIMAL_LABELS)
    long["year"] = meta["year"].astype(int)
    long["adjusted"] = meta["adj"].notna()
    return long.drop(columns="col")

inv = pd.concat([tidy_inventory(d) for d in frames.values()], ignore_index=True)
inv = add_fips(inv)
inv = inv[inv["state"] == STATE].copy()  # project scope: Iowa only
inv["head_count"] = pd.to_numeric(inv["head_count"], errors="coerce")

INV_COLS = ["state_fips", "county_fips", "county_name", "state",
            "year", "animal", "head_count", "adjusted"]
inv_clean = inv[INV_COLS].sort_values(
    ["county_fips", "year", "animal"]
).reset_index(drop=True)
assert not inv_clean.duplicated(["county_fips", "year", "animal"]).any()
print(f"Inventory table: {len(inv_clean):,} rows")
inv_clean.head()

Inventory table: 14,850 rows


,state_fips,county_fips,county_name,state,year,animal,head_count,adjusted
0,19,19001,Adair,IA,1950,all cattle and calves,52030.0,True
1,19,19001,Adair,IA,1950,beef cows,11894.0,True
2,19,19001,Adair,IA,1950,broilers,2240.0,True
3,19,19001,Adair,IA,1950,hogs and pigs,98957.0,True
4,19,19001,Adair,IA,1950,horses and ponies,3342.0,True


## Step 5 - Weight-coefficient reference

The `coef` tab gives, per year, each animal group's USDA average live slaughter weight and the coefficient of change vs. the 1992 base year. Tidy it into `(year, animal, weight_coef, usda_live_weight_avg)`.

In [6]:
coef = xl.parse("coef").rename(
    columns={"Federally inspected average live slaughter weight in pounds": "year"})
coef = coef[coef["year"].apply(lambda v: str(v).strip().isdigit())].copy()
coef["year"] = coef["year"].astype(int)

# usda-live-weight column uses "sheep-lambs"; coef column uses "sheep"
COEF_ANIMALS = {"cattle": "cattle", "hogs": "hogs", "chickens": "chickens",
                "broilers": "broilers", "turkeys": "turkeys", "sheep": "sheep-lambs"}
parts = []
for animal, wcol in COEF_ANIMALS.items():
    parts.append(pd.DataFrame({
        "year": coef["year"],
        "animal": animal,
        "weight_coef": coef[f"{animal}-weight-coef"],
        "usda_live_weight_avg": coef[f"usda-live-weight-{wcol}-avg"],
    }))
coef_clean = pd.concat(parts, ignore_index=True).sort_values(
    ["year", "animal"]).reset_index(drop=True)
assert not coef_clean.duplicated(["year", "animal"]).any()
print(f"Coefficient table: {len(coef_clean):,} rows, years "
      f"{coef_clean.year.min()}-{coef_clean.year.max()}")
coef_clean.head()

Coefficient table: 90 rows, years 1950-2017


,year,animal,weight_coef,usda_live_weight_avg
0,1950,broilers,0.743538,3.355833
1,1950,cattle,0.868851,1018.800000
2,1950,chickens,1.039562,4.905000
3,1950,hogs,0.941897,238.300000
4,1950,sheep,0.780237,98.700000


## Step 6 - Sanity check

In [7]:
print("Iowa (state IA) total manure-N by year, kg (animal_category == \"Total\"):")
ia = nut_clean[(nut_clean.state == "IA") & (nut_clean.animal_category == "Total")
               & (nut_clean.nutrient == "N")]
print(ia.groupby("year")["value_kg"].sum().round(0).to_string())
print(f"\nInventory adjusted (back-filled) share: "
      f"{inv_clean.adjusted.mean():.1%} of {len(inv_clean):,} rows")

Iowa (state IA) total manure-N by year, kg (animal_category == "Total"):
year
1950    323989312.0
1954    395841437.0
1959    424836511.0
1964    434070908.0
1969    412161440.0
1974    389362515.0
1978    400230411.0
1982    389676047.0
1987    316278049.0
1992    322948711.0
1997    318126189.0
2002    334478170.0
2007    397204024.0
2012    420285275.0
2017    463284127.0

Inventory adjusted (back-filled) share: 90.0% of 14,850 rows


## Step 7 - Save

In [8]:
outputs = {
    "np-manure-clean.csv": nut_clean,
    "manure-animal-inventory-clean.csv": inv_clean,
    "manure-weight-coefficients-clean.csv": coef_clean,
}
for fname, frame in outputs.items():
    out_file = CLEAN_DIR / fname
    frame.to_csv(out_file, index=False)
    print(f"Saved {len(frame):,} rows -> {out_file}")

Saved 14,850 rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture/np-manure-clean.csv
Saved 14,850 rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture/manure-animal-inventory-clean.csv
Saved 90 rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture/manure-weight-coefficients-clean.csv
